<a href="https://colab.research.google.com/github/RajChauhan2911/Image-Classification-of-Vehicles-/blob/main/image_classifier_logistic_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install scikit-learn pillow numpy matplotlib --quiet

In [ ]:
import os
import zipfile
import warnings

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, UnidentifiedImageError

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    ConfusionMatrixDisplay
)
from google.colab import files

warnings.filterwarnings("ignore")
print("✅ All libraries imported successfully!")

✅ All libraries imported successfully!


In [ ]:
IMG_SIZE      = (64, 64)


USE_GRAYSCALE = True


TEST_SPLIT    = 0.2

RANDOM_SEED   = 42

LR_C          = 1.0

print(f"📐 Image size   : {IMG_SIZE}")
print(f"🎨 Mode         : {'Grayscale' if USE_GRAYSCALE else 'RGB'}")
print(f"🔀 Test split   : {int(TEST_SPLIT*100)}%")

📐 Image size   : (64, 64)
🎨 Mode         : Grayscale
🔀 Test split   : 20%


In [ ]:
print("📤 Please upload your dataset ZIP file...")
uploaded = files.upload()

if not uploaded:
    raise ValueError("❌ No file uploaded. Please run this cell again and select your ZIP file.")

zip_filename = list(uploaded.keys())[0]
print(f"\n📦 Received: {zip_filename}")

In [ ]:
EXTRACT_DIR = "dataset"

with zipfile.ZipFile(zip_filename, "r") as zf:
    zf.extractall(EXTRACT_DIR)

print(f"✅ Extracted to: '{EXTRACT_DIR}/'")

def find_dataset_root(base_dir):
    """Walk until we find a folder containing sub-folders (class dirs).
    Skips hidden and __MACOSX folders automatically.
    """
    candidate_root = base_dir
    while True:
        contents = [d for d in os.listdir(candidate_root) if not d.startswith(('.', '__'))]
        sub_dirs = [d for d in contents if os.path.isdir(os.path.join(candidate_root, d))]


        if len(sub_dirs) == 1 and not any(os.path.isfile(os.path.join(candidate_root, f)) for f in contents):
            candidate_root = os.path.join(candidate_root, sub_dirs[0])
        else:
            break
    return candidate_root

DATASET_ROOT = find_dataset_root(EXTRACT_DIR)

CLASS_LABELS = sorted([
    d for d in os.listdir(DATASET_ROOT)
    if os.path.isdir(os.path.join(DATASET_ROOT, d))
    and not d.startswith(('.', '__'))
])

if not CLASS_LABELS:
    raise ValueError("❌ No class folders found. Make sure your ZIP has sub-folders named after classes.")

print(f"\n📂 Dataset root : {DATASET_ROOT}")
print(f"🏷️  Classes found: {CLASS_LABELS}")
print(f"📊 Total classes: {len(CLASS_LABELS)}")

In [ ]:
VALID_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".webp"}

def preprocess_image(img_path, img_size=IMG_SIZE, grayscale=USE_GRAYSCALE):
    """
    Load one image → resize → grayscale/RGB → normalise → flatten.
    Returns a 1-D numpy array, or None if the file is invalid.
    """
    ext = os.path.splitext(img_path)[1].lower()
    if ext not in VALID_EXTENSIONS:
        return None
    try:
        img = Image.open(img_path)
        img = img.convert("L" if grayscale else "RGB")
        img = img.resize(img_size, Image.LANCZOS)
        arr = np.array(img, dtype=np.float32) / 255.0
        return arr.flatten()
    except (UnidentifiedImageError, Exception):
        return None

print("✅ Preprocessing function ready.")

✅ Preprocessing function ready.


In [ ]:
import os

X, y = [], []
skipped = 0

for label in CLASS_LABELS:
    class_dir = os.path.join(DATASET_ROOT, label)
    loaded = 0
    if not os.path.isdir(class_dir):
        print(f"  [{label}]  →  Skipping '{class_dir}' (not a directory or does not exist).")
        skipped += len(os.listdir(class_dir)) if os.path.exists(class_dir) else 0
        continue

    for fname in os.listdir(class_dir):
        fpath = os.path.join(class_dir, fname)

        if not os.path.isfile(fpath):
            skipped += 1
            continue

        vec   = preprocess_image(fpath)
        if vec is not None:
            X.append(vec)
            y.append(label)
            loaded += 1
        else:
            skipped += 1
    print(f"  [{label}]  →  {loaded} images loaded")

X = np.array(X)
y = np.array(y)

channels = 1 if USE_GRAYSCALE else 3
print(f"\n✅ Total loaded  : {len(X)} images")
print(f"⚠️  Skipped       : {skipped} invalid files")

if len(X) > 0:
    print(f"📐 Feature size  : {X.shape[1]}  "
          f"({'grayscale' if USE_GRAYSCALE else 'RGB'}  "
          f"{IMG_SIZE[0]}×{IMG_SIZE[1]}×{channels})")
elif len(X) == 0 and CLASS_LABELS:
    print("❌ No valid images were loaded. Feature size cannot be determined.")
    print("   Please check your dataset structure. It appears that 'CLASS_LABELS' might be pointing to directories ")
    print("   that contain *other* class directories, rather than directly containing image files.")
    raise RuntimeError("No valid images were loaded. Please review your dataset structure and 'CLASS_LABELS' definition.")

In [ ]:
le    = LabelEncoder()
y_enc = le.fit_transform(y)

print("🔢 Label encoding:")
for cls, idx in zip(le.classes_, le.transform(le.classes_)):
    print(f"   {idx}  →  {cls}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc,
    test_size=TEST_SPLIT,
    random_state=RANDOM_SEED,
    stratify=y_enc
)

print(f"\n✅ Train samples : {len(X_train)}")
print(f"✅ Test  samples : {len(X_test)}")

In [ ]:
n_samples = min(12, len(X))
indices   = np.random.choice(len(X), n_samples, replace=False)
cols = 6
rows = (n_samples + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(cols * 2.2, rows * 2.4))
axes = axes.flatten()

for i, idx in enumerate(indices):
    vec   = X[idx]
    label = y[idx]
    img   = vec.reshape(IMG_SIZE) if USE_GRAYSCALE else vec.reshape((*IMG_SIZE, 3))
    axes[i].imshow(img, cmap="gray" if USE_GRAYSCALE else None, vmin=0, vmax=1)
    axes[i].set_title(label, fontsize=9, fontweight="bold")
    axes[i].axis("off")

for j in range(len(indices), len(axes)):
    axes[j].axis("off")

plt.suptitle("📸 Sample Images from Dataset", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
print("🚀 Training Logistic Regression... (may take a moment)")

model = LogisticRegression(
    max_iter=1000,
    solver="lbfgs",
    multi_class="auto",
    C=LR_C,
    random_state=RANDOM_SEED,
    n_jobs=-1
)

model.fit(X_train, y_train)
print("✅ Training complete!")

In [ ]:
y_pred = model.predict(X_test)
acc    = accuracy_score(y_test, y_pred)

print(f"🎯 Test Accuracy : {acc * 100:.2f}%\n")
print("📋 Classification Report:")
print(classification_report(y_test, y_pred, target_names=le.classes_))

In [ ]:
size = max(4, len(CLASS_LABELS))
fig, ax = plt.subplots(figsize=(size, size))

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=le.classes_,
    cmap="Blues",
    ax=ax
)
ax.set_title("Confusion Matrix", fontsize=14, fontweight="bold", pad=15)
plt.tight_layout()
plt.show()

In [ ]:
n      = min(12, len(X_test))
idxs   = np.random.choice(len(X_test), n, replace=False)
cols   = 4
rows   = (n + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3.2))
axes = axes.flatten()

for i, idx in enumerate(idxs):
    vec   = X_test[idx]
    true  = le.classes_[y_test[idx]]
    pred  = le.classes_[model.predict([vec])[0]]
    color = "#2ecc71" if true == pred else "#e74c3c"
    img   = vec.reshape(IMG_SIZE) if USE_GRAYSCALE else vec.reshape((*IMG_SIZE, 3))
    axes[i].imshow(img, cmap="gray" if USE_GRAYSCALE else None, vmin=0, vmax=1)
    axes[i].set_title(f"True: {true}\nPred: {pred}", color=color, fontsize=9, fontweight="bold")
    axes[i].axis("off")

for j in range(n, len(axes)):
    axes[j].axis("off")

plt.suptitle("Sample Predictions  (🟢 correct  /  🔴 wrong)",
             fontsize=12, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
print("🖼️  Upload an image to classify...")
uploaded_img = files.upload()

if not uploaded_img:
    print("❌ No image uploaded.")
else:
    for img_filename in uploaded_img.keys():
        print(f"\n🔍 Processing: {img_filename}")

        vec = preprocess_image(img_filename)

        if vec is None:
            print("❌ Could not read image.\n"
                  "   Make sure it is a valid JPEG / PNG / BMP file.")
            continue

        pred_idx   = model.predict([vec])[0]
        pred_label = le.classes_[pred_idx]
        proba      = model.predict_proba([vec])[0]

        print(f"\n✅ Predicted class : {pred_label.upper()}")
        print("\n📊 Confidence scores:")
        for cls, prob in zip(le.classes_, proba):
            bar   = "█" * int(prob * 30)
            arrow = " ◀" if cls == pred_label else ""
            print(f"   {cls:<20} {prob*100:5.1f}%  {bar}{arrow}")

        fig, (ax_img, ax_bar) = plt.subplots(1, 2, figsize=(10, 4))

        display_img = Image.open(img_filename).convert("L" if USE_GRAYSCALE else "RGB")
        ax_img.imshow(display_img, cmap="gray" if USE_GRAYSCALE else None)
        ax_img.set_title(f"Predicted: {pred_label}",
                         fontsize=14, fontweight="bold", color="#2c3e50")
        ax_img.axis("off")

        colours = ["#2ecc71" if cls == pred_label else "#95a5a6"
                   for cls in le.classes_]
        bars = ax_bar.barh(le.classes_, proba * 100, color=colours,
                           edgecolor="white", height=0.6)
        ax_bar.set_xlim(0, 100)
        ax_bar.set_xlabel("Confidence (%)", fontsize=11)
        ax_bar.set_title("Class Probabilities", fontsize=12, fontweight="bold")
        ax_bar.bar_label(bars, fmt="%.1f%%", padding=3, fontsize=10)
        ax_bar.spines[["top", "right"]].set_visible(False)

        plt.tight_layout()
        plt.show()